# Import and Configurations

In [ ]:
# ============================================================
# IMPORTS
# ============================================================

import os
import random
import joblib
import warnings

import numpy as np
import pandas as pd
import scanpy as sc

from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier


warnings.filterwarnings("ignore")


# ============================================================
# SETTINGS
# ============================================================

# ------------------------------------------------------------
# Input h5ad
# ------------------------------------------------------------

DATA_PATH = (
    "sn_data.h5ad"
)


# ------------------------------------------------------------
# Cell-type column
# ------------------------------------------------------------

LABEL_COLUMN = "cell_type"


# ------------------------------------------------------------
# Number of cells per cell type
# ------------------------------------------------------------

CELLS_PER_TYPE = 50


# ------------------------------------------------------------
# Random Forest settings
# ------------------------------------------------------------

N_ESTIMATORS = 300

N_JOBS = -1


# ------------------------------------------------------------
# Base random seed
# ------------------------------------------------------------

BASE_SEED = 42


# ------------------------------------------------------------
# Output directory
# ------------------------------------------------------------

OUTPUT_DIR = (
    "./LOCTO_random_forest_91_models"
)



os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)



# Data Prep

In [ ]:
# ============================================================
# LOAD DATA
# ============================================================

print("\n")
print("=" * 70)
print("LOADING DATA")
print("=" * 70)

adata = sc.read_h5ad(
    DATA_PATH
)

print(
    f"Number of cells: {adata.n_obs}"
)

print(
    f"Number of genes/features: {adata.n_vars}"
)


# ============================================================
# CHECK LABEL COLUMN
# ============================================================

if LABEL_COLUMN not in adata.obs.columns:

    raise ValueError(
        f"Column '{LABEL_COLUMN}' was not found in adata.obs."
    )


# ============================================================
#  GET CELL TYPES
# ============================================================

cell_type_series = (
    adata.obs[LABEL_COLUMN]
    .astype(str)
)

cell_types = sorted(
    cell_type_series.unique()
)

n_celltypes = len(
    cell_types
)


print(
    f"\nNumber of cell types: {n_celltypes}"
)

print("\nCell types:")

for i, ct in enumerate(cell_types, start=1):

    n_cells = (
        cell_type_series == ct
    ).sum()

    print(
        f"{i:3d}. {ct:40s} "
        f"{n_cells} cells"
    )


# ============================================================
# CHECK WHETHER EVERY CELL TYPE HAS 50 CELLS
# ============================================================

cell_type_counts = (
    cell_type_series
    .value_counts()
    .sort_index()
)

insufficient_types = (
    cell_type_counts[
        cell_type_counts < CELLS_PER_TYPE
    ]
)

if len(insufficient_types) > 0:

    print("\n")
    print("=" * 70)
    print("WARNING")
    print("=" * 70)

    print(
        "The following cell types have fewer than "
        f"{CELLS_PER_TYPE} cells:"
    )

    print(
        insufficient_types
    )

    print(
        "\nThose cell types cannot contribute 50 cells "
        "to the models."
    )

    raise ValueError(
        "At least one cell type has fewer than 50 cells."
    )


# ============================================================
# CONVERT X TO NUMPY
# ============================================================

print("\n")
print("=" * 70)
print("PREPARING FEATURE MATRIX")
print("=" * 70)


# ------------------------------------------------------------
# Handle sparse matrices
# ------------------------------------------------------------

if hasattr(
    adata.X,
    "toarray"
):

    X = adata.X.toarray()

else:

    X = np.asarray(
        adata.X
    )


X = np.asarray(
    X,
    dtype=np.float32
)


print(
    f"Feature matrix shape: {X.shape}"
)


# ============================================================
# 9. ORIGINAL CELL INDICES
# ============================================================

original_indices = np.arange(
    adata.n_obs
)


# ============================================================
# PRE-COMPUTE INDICES FOR EACH CELL TYPE
# ============================================================

cell_type_indices = {}

for ct in cell_types:

    indices = np.where(
        cell_type_series.values == ct
    )[0]

    cell_type_indices[ct] = indices


# ============================================================
# SUMMARY TABLE
# ============================================================

model_summary = []



# ML Model Training

In [2]:
# ============================================================
# 12. TRAIN 92 LEAVE-ONE-CELL-TYPE-OUT MODELS
# ============================================================

print("\n")
print("=" * 70)
print("STARTING LEAVE-ONE-CELL-TYPE-OUT TRAINING")
print("=" * 70)


for model_number, held_out_cell_type in enumerate(
    cell_types,
    start=1
):


    # ========================================================
    # MODEL-SPECIFIC SEED
    # ========================================================

    seed = (
        BASE_SEED +
        model_number
    )

    np.random.seed(
        seed
    )

    random.seed(
        seed
    )


    # ========================================================
    # MODEL DIRECTORY
    # ========================================================

    # Make folder name safe for filesystem
    safe_name = (
        held_out_cell_type
        .replace("/", "_")
        .replace("\\", "_")
        .replace(" ", "_")
    )

    model_dir = os.path.join(
        OUTPUT_DIR,
        f"{model_number:03d}_leave_out_{safe_name}"
    )

    os.makedirs(
        model_dir,
        exist_ok=True
    )


    # ========================================================
    # PRINT HEADER
    # ========================================================

    print("\n")
    print("=" * 70)

    print(
        f"MODEL {model_number}/{n_celltypes}"
    )

    print(
        f"Held-out cell type: {held_out_cell_type}"
    )

    print("=" * 70)


    # ========================================================
    # SELECT 50 CELLS FROM EVERY OTHER CELL TYPE
    # ========================================================

    training_indices_list = []

    training_cell_types = []


    for training_cell_type in cell_types:


        # ----------------------------------------------------
        # IMPORTANT:
        # Skip held-out cell type completely
        # ----------------------------------------------------

        if training_cell_type == held_out_cell_type:

            continue


        # ----------------------------------------------------
        # Available cells
        # ----------------------------------------------------

        available_indices = (
            cell_type_indices[
                training_cell_type
            ]
        )


        # ----------------------------------------------------
        # Sample exactly 50
        # ----------------------------------------------------

        rng = np.random.default_rng(
            seed +
            hash(training_cell_type) % 100000
        )


        selected_indices = rng.choice(
            available_indices,
            size=CELLS_PER_TYPE,
            replace=False
        )


        training_indices_list.extend(
            selected_indices.tolist()
        )


        training_cell_types.extend(
            [training_cell_type] *
            CELLS_PER_TYPE
        )


    # ========================================================
    # CONVERT TO ARRAYS
    # ========================================================

    training_indices = np.asarray(
        training_indices_list,
        dtype=int
    )

    training_labels = np.asarray(
        training_cell_types,
        dtype=str
    )


    # ========================================================
    # SHUFFLE TRAINING DATA
    # ========================================================

    shuffle_rng = np.random.default_rng(
        seed
    )

    shuffle_order = (
        shuffle_rng.permutation(
            len(training_indices)
        )
    )

    training_indices = (
        training_indices[
            shuffle_order
        ]
    )

    training_labels = (
        training_labels[
            shuffle_order
        ]
    )


    # ========================================================
    # CREATE TRAINING MATRIX
    # ========================================================

    X_train = X[
        training_indices
    ]




    label_encoder = LabelEncoder()

    y_train = (
        label_encoder.fit_transform(
            training_labels
        )
    )


    # ========================================================
    # VERIFY HELD-OUT TYPE IS ABSENT
    # ========================================================

    if (
        held_out_cell_type
        in label_encoder.classes_
    ):

        raise RuntimeError(
            "ERROR: Held-out cell type "
            "appeared in the label encoder!"
        )


    # ========================================================
    # NUMBER OF TRAINING CLASSES
    # ========================================================

    n_training_classes = len(
        label_encoder.classes_
    )


    # ========================================================
    # CLASSES
    # ========================================================

    expected_classes = (
        n_celltypes - 1
    )

    if (
        n_training_classes
        != expected_classes
    ):

        raise RuntimeError(
            f"Expected {expected_classes} "
            f"training classes but found "
            f"{n_training_classes}."
        )


    # ========================================================
    # TRAIN RANDOM FOREST
    # ========================================================

    print(
        "\nTraining Random Forest..."
    )

    model = RandomForestClassifier(

        n_estimators=N_ESTIMATORS,

        random_state=seed,

        n_jobs=N_JOBS

    )


    model.fit(
        X_train,
        y_train
    )


    print(
        "Training complete."
    )


    # ========================================================
    # SAVE RANDOM FOREST
    # ========================================================

    model_path = os.path.join(
        model_dir,
        "RANDOM_FOREST_MODEL.joblib"
    )

    joblib.dump(
        model,
        model_path
    )


    # ========================================================
    # SAVE LABEL ENCODER
    # ========================================================

    encoder_path = os.path.join(
        model_dir,
        "LABEL_ENCODER.joblib"
    )

    joblib.dump(
        label_encoder,
        encoder_path
    )


    # ========================================================
    # SAVE TRAINING CELL INDICES
    # ========================================================

    training_cells_df = pd.DataFrame({

        "Original_Cell_Index":
            training_indices,

        "Cell_Type":
            training_labels

    })


    training_cells_path = os.path.join(
        model_dir,
        "TRAINING_CELLS.csv"
    )

    training_cells_df.to_csv(
        training_cells_path,
        index=False
    )


    # ========================================================
    # SAVE CLASS ORDER
    # ========================================================

    class_order_df = pd.DataFrame({

        "Encoded_Label":
            np.arange(
                len(
                    label_encoder.classes_
                )
            ),

        "Cell_Type":
            label_encoder.classes_

    })


    class_order_path = os.path.join(
        model_dir,
        "CLASS_ORDER.csv"
    )

    class_order_df.to_csv(
        class_order_path,
        index=False
    )


    # ========================================================
    # SAVE MODEL INFORMATION
    # ========================================================

    model_info = {

        "Model_Number":
            model_number,

        "Held_Out_Cell_Type":
            held_out_cell_type,

        "Number_Total_Cell_Types":
            n_celltypes,

        "Number_Training_Cell_Types":
            n_training_classes,

        "Cells_Per_Training_Cell_Type":
            CELLS_PER_TYPE,

        "Total_Training_Cells":
            len(training_indices),

        "N_Estimators":
            N_ESTIMATORS,

        "Random_Seed":
            seed,

        "Label_Column":
            LABEL_COLUMN,

        "Number_Features":
            X.shape[1],

        "Held_Out_Cell_Type_In_Training":
            False

    }


    info_df = pd.DataFrame(
        [model_info]
    )


    info_path = os.path.join(
        model_dir,
        "MODEL_INFO.csv"
    )

    info_df.to_csv(
        info_path,
        index=False
    )


    # ========================================================
    # SAVE MODEL SUMMARY
    # ========================================================

    model_summary.append({

        "Model_Number":
            model_number,

        "Held_Out_Cell_Type":
            held_out_cell_type,

        "Training_Cell_Types":
            n_training_classes,

        "Cells_Per_Type":
            CELLS_PER_TYPE,

        "Total_Training_Cells":
            len(training_indices),

        "N_Estimators":
            N_ESTIMATORS,

        "Random_Seed":
            seed,

        "Model_Path":
            model_path

    })




# ============================================================
# SAVE GLOBAL SUMMARY
# ============================================================

summary_df = pd.DataFrame(
    model_summary
)


summary_path = os.path.join(
    OUTPUT_DIR,
    "MODELS_SUMMARY.csv"
)


summary_df.to_csv(
    summary_path,
    index=False
)


# ============================================================
# SAVE ALL CELL-TYPE LIST
# ============================================================

all_celltypes_df = pd.DataFrame({

    "Model_Number":
        np.arange(
            1,
            n_celltypes + 1
        ),

    "Held_Out_Cell_Type":
        cell_types

})


all_celltypes_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "ALL_HELD_OUT_CELL_TYPES.csv"
    ),
    index=False
)




